In [ ]:
## Source: https://graphacademy.neo4j.com/courses/llm-fundamentals/3-intro-to-langchain/1-langchain/
## This example program uses Langchain to build a chatbot that answers questions about Neo4j Cypher queries.
## The program interacts with an OpenAI LLM, uses a prompt template to instruct the LLM on how to act, and uses a memory component to retain context and store the history in Neo4j.

In [1]:
!pip install langchain
!pip install openai
!pip install neo4j

!pip install langchain-community langchain-neo4j langchainhub neo4j

!pip install langchain_openai
!pip install langchain_neo4j

import os

from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_react_agent
from langchain.tools import Tool
from langchain import hub
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain.schema import StrOutputParser
from langchain_neo4j import Neo4jChatMessageHistory, Neo4jGraph
from uuid import uuid4

SESSION_ID = str(uuid4())
print(f"Session ID: {SESSION_ID}")

# Store your API key in a variable called 'openai_api_key'
openai_api_key = "YOUR_OPENAI_API_KEY_HERE"  # Replace with your actual API key


llm = ChatOpenAI(
    openai_api_key=openai_api_key # Pass the API key directly
    )

graph = Neo4jGraph(
    url="bolt://YOUR_NEO4J_HOST:7687",
    username="neo4j",
    password="YOUR_NEO4J_PASSWORD"
)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a Neo4j expert having a conversation about how to create Cypher queries",
        ),
        ("human", "{input}"),
    ]
)

cypher_chat = prompt | llm | StrOutputParser()

def get_memory(session_id):
    return Neo4jChatMessageHistory(session_id=session_id, graph=graph)

tools = [
    Tool.from_function(
        name="Cypher Support",
        description="For when you need to talk about Cypher queries.",
        func=cypher_chat.invoke,
    )
]

agent_prompt = hub.pull("hwchase17/react-chat")
agent = create_react_agent(llm, tools, agent_prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools)

cypher_agent = RunnableWithMessageHistory(
    agent_executor,
    get_memory,
    input_messages_key="input",
    history_messages_key="chat_history",
)

while (q := input("> ")) != "exit":

    response = cypher_agent.invoke(
        {
            "input": q
        },
        {"configurable": {"session_id": SESSION_ID}},
    )

    print(response["output"])

Session ID: 43de9f52-8fd5-43c5-b0aa-9680d6d1646d


/usr/local/lib/python3.11/dist-packages/langsmith/client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


> How do you query movies?


To query movies in Neo4j using Cypher, you can use a simple MATCH statement. If you want to retrieve all movies, you can use `MATCH (m:Movie) RETURN m`.
> how do you query movies directed by a director
You can query movies directed by a director in Neo4j using the Cypher query: MATCH (d:Director)-[:DIRECTED]->(m:Movie) RETURN d, m. This will return all pairs of Directors and Movies that are connected by the DIRECTED relationship.
> How do you query movies of a particular genre?
To query movies of a particular genre in Neo4j using Cypher, you can use the following query:

MATCH (m:Movie)-[:IN_GENRE]->(g:Genre)
WHERE g.name = 'Action'
RETURN m

In this query:
- MATCH (m:Movie)-[:IN_GENRE]->(g:Genre) matches movies connected to genres through the IN_GENRE relationship.
- WHERE g.name = 'Action' filters the results to only include movies with the genre name 'Action'.
- RETURN m returns the matched movies.

You can replace 'Action' with the genre of your choice to query movies of a differen